In [ ]:
import warnings
import glob
import os
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

from tqdm import tqdm
from utils.model_fitting import sigmoid,sigmoid_fun,slope_log,fit_sigmoid
from utils.plots import *
from biophysical_model.dopamine_toolbox import initialize_config_drugs
from utils.path import PathConfig
warnings.filterwarnings("ignore")
rseed =10
np.random.seed(rseed)




- `res['da']`       : DA concentrarion (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
- `res['d1_occ']`   : D1 occupancy (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
- `res['d2l_occ']`  : D2l occupancy(Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
- `res['d2s_occ']`  : D2s occupancy(Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
- `res['d1_act']`   : D1 activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
- `res['d2l_act']`  : D2l activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
- `res['d2s_act']`  :  D2s activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)

Where: 
- N Drug:  number of drug concentrations tested 
- N DA : number of dopamine baseline levels (derived from baseline firing rates from single neurons)
- N efficacy D2l: number of D2l receptor efficacies 
- N efficacy D2s: number of D2s receptor efficacies 
- N Timestamps : NUmber of timestamps per trial 

## Reproduction of drug manipulations

- This code plots the results from the biophysical simulations performed by adding an additional D2 receptor agonist (bromocriptine)
- The simulations were done to reproduce the effects of bromocriptine on reversal learning in the study: **Cools, R., Frank M.J., Gibbs S., Miyakawa A., Jagust W. & D'Esposito M.  Striatal dopamine predicts outcome-specific reversal learning and its sensitivity to dopaminergic drug administration. J. Neurosci. 29, 1538–1543 (2009)** in which healthy humans were given this drug and asked to perform a reversal task
- The files of the simnulations are dictionaries, containing:
    - `res['da']`       : DA concentrarion (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
    - `res['d1_occ']`   : D1 occupancy (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
    - `res['d2l_occ']`  : D2l occupancy(Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
    - `res['d2s_occ']`  : D2s occupancy(Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
    - `res['d1_act']`   : D1 activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps)
    - `res['d2l_act']`  : D2l activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)
    - `res['d2s_act']`  :  D2s activation (Size: N Drug, N DA, N efficacy D2l, N efficacy D2s,N Timestamps, N agonists)

    Where: 
    - N Drug:  number of drug concentrations tested 
    - N DA : number of dopamine baseline levels (derived from baseline firing rates from single neurons)
    - N efficacy D2l: number of D2l receptor efficacies 
    - N efficacy D2s: number of D2s receptor efficacies 
    - N Timestamps : NUmber of timestamps per trial 


In [ ]:
#%% Load single unit simulations
# couldn't load:
# r_K_2012-08-11_09-57-32_TT2_1_results_only_base.
# r_K_2012-08-15_13-43-17_TT6_2_results_only_base.npy
# r_K_2012-08-07_13-32-52_TT1_1_results_only_base.npy
# r_HeJ_2013-06-24_15-36-07_TT1_1_results_only_base.npy'

paths = PathConfig()
g_dir = paths.g2_dir
data_path = os.path.join(g_dir,'drug_experiments')
file_list = glob.glob(data_path + '/*_results_only_base.npy')
res_mats = []
for id_unit in tqdm(range(len(file_list))):
    res_mats.append(np.load(file_list[id_unit],allow_pickle=True).item())
drug_concv, efficacies, ndrug, neffs, id_bins_base,da_base,rec_base,log_delta,ec50_d1,ec50_d2,da_eta = initialize_config_drugs()
nunits = len(res_mats)
nda = res_mats[0]['da'].shape[1]
nda_base =len(da_base)
n_points = len(da_eta)
efficacies_p = np.ceil(efficacies*10)/10

### Dose-occupancy curve in the presence of bromocriptine
The following code fits a sigmoid function to compute the dose-occupancy curves of the D2 recptors in the presence of difference concentrations (color) and efficacies of the dopamine agonist bromocriptine 

In [30]:
#%% Fit sigmoid to the curves for d2 activation: 
ec50_act, bias_act = np.zeros((neffs,neffs,ndrug)), np.zeros((neffs,neffs,ndrug))
ec50_occ, bias_occ = np.zeros((neffs,neffs,ndrug)), np.zeros((neffs,neffs,ndrug))
popts_occ,popts_act = np.zeros((neffs,neffs,ndrug,4)), np.zeros((neffs,neffs,ndrug,4))
mus_da,mus_occ,mus_act =  np.zeros((neffs,neffs,ndrug,nda)),np.zeros((neffs,neffs,ndrug,nda)),np.zeros((neffs,neffs,ndrug,nda))
yfits_occ,yfits_act =  np.zeros((neffs,neffs,ndrug,nda_base)),np.zeros((neffs,neffs,ndrug,nda_base))

for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        for idr,drug_conc in zip(range(ndrug),drug_concv):
            temp_act,temp_occ,temp_da = np.nan*np.ones((nda,nunits)),np.nan*np.ones((nda,nunits)),np.nan*np.ones((nda,nunits))
            for id_unit in range(nunits):
                temp_da[:,id_unit] = np.nanmean(res_mats[id_unit]['da'][0,:,i_eff_d2l,i_eff_d2s,id_bins_base],axis=0)
                temp_act[:,id_unit] = np.nanmean( res_mats[id_unit]['d2l_act'][idr,:,i_eff_d2l,i_eff_d2s,id_bins_base],axis=0)
                temp_occ[:,id_unit] = np.nanmean(np.sum(res_mats[id_unit]['d2l_occ'][idr,:,i_eff_d2l,i_eff_d2s,id_bins_base],axis=2),axis=0)   
            mu_act,mu_occ,mu_da = np.nanmean(temp_act,axis=1) ,np.nanmean(temp_occ,axis=1) ,np.nanmean(temp_da,axis=1) 
            mus_da[i_eff_d2l,i_eff_d2s,idr,:],mus_occ[i_eff_d2l,i_eff_d2s,idr,:],mus_act[i_eff_d2l,i_eff_d2s,idr,:] = mu_da,mu_occ,mu_act
            popt_occ, _,yfit_occ = fit_sigmoid(mu_da,mu_occ,da_base,log_xscale=True)
            popt_act, _,yfit_act = fit_sigmoid(mu_da,mu_act,da_base,log_xscale=True)
            yfits_occ[i_eff_d2l,i_eff_d2s,idr,:],yfits_act[i_eff_d2l,i_eff_d2s,idr,:] = yfit_occ,yfit_act
            bias_act[i_eff_d2l,i_eff_d2s,idr] = popt_act[-1]
            popts_act[i_eff_d2l,i_eff_d2s,idr,:] = popt_act
            bias_occ[i_eff_d2l,i_eff_d2s,idr] = popt_occ[-1]
            ec50_occ[i_eff_d2l,i_eff_d2s,idr] = popt_occ[1]
            popts_occ[i_eff_d2l,i_eff_d2s,idr,:] = popt_occ

IndexError: list index out of range


### Figure 7 & Extended Data Figure 6
Plots of the dose-occupancy curves of the D2 recptors in the presence of difference concentrations (color) and efficacies of the dopamine agonist bromocriptine 

In [ ]:
#%% Plot sigmoid to the curves for d2 activation: 
__,axo = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))
__,axa = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))

for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        axo[i_eff_d2l,i_eff_d2s].plot(da_base,rec_base,color='grey')
        axa[i_eff_d2l,i_eff_d2s].plot(da_base,rec_base,color='grey')
        for idr,drug_conc in zip(range(ndrug),drug_concv):
            mu_da, mu_occ, mu_act = mus_da[i_eff_d2l,i_eff_d2s,idr,:],mus_occ[i_eff_d2l,i_eff_d2s,idr,:],mus_act[i_eff_d2l,i_eff_d2s,idr,:]
            yfit_occ, yfit_act = yfits_occ[i_eff_d2l,i_eff_d2s,idr,:],yfits_act[i_eff_d2l,i_eff_d2s,idr,:]
            axo[i_eff_d2l,i_eff_d2s].plot(np.log10(mu_da),mu_occ,'o',color=cm.viridis(idr/len(drug_concv)) )
            axo[i_eff_d2l,i_eff_d2s].plot(da_base,yfit_occ,color=cm.viridis(idr/len(drug_concv)) )
            axa[i_eff_d2l,i_eff_d2s].plot(np.log10(mu_da),mu_act,'o',color=cm.viridis(idr/len(drug_concv)) )
            axa[i_eff_d2l,i_eff_d2s].plot(da_base,yfit_act,color=cm.viridis(idr/len(drug_concv)) )
        plot_config(axo[i_eff_d2l,i_eff_d2s],'log[DA] nM','D2l occ',9,False)
        plot_config(axa[i_eff_d2l,i_eff_d2s],'log[DA] nM','D2l act',9,False)
        axo[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
        axa[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)


### Receptor sensitivities and asymmetric scaling factor in the presence of bromocriptine
The following code computes:
- Receptor sensitivities of D1 and D2 receptors (slopes of the dose-occupancy curve given a `log_delta` transient in dopamine )
- Asymmetric scaling factor $\tau$ given by the receptor sensitivities

The parameters were computed in the presence of difference concentrations and efficacies of the dopamine agonist bromocriptine , as well as for different levels of baseline dopamine concentration

In [ ]:

d1_slope = np.zeros((neffs,neffs,ndrug+1,n_points))
d2_slope_act = np.zeros((neffs,neffs,ndrug+1,n_points))
eta_act = np.zeros((neffs,neffs,ndrug+1,n_points))
d2_slope_occ = np.zeros((neffs,neffs,ndrug+1,n_points))
eta_occ = np.zeros((neffs,neffs,ndrug+1,n_points))
for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        for idr,idrug in enumerate(np.arange(-1,ndrug)):
            d1temp,d2temp_act,d2temp_occ =np.zeros(n_points),np.zeros(n_points),np.zeros(n_points)
            for id  in np.arange(n_points):
                if idrug>-1:
                    popt_act = popts_act[i_eff_d2l,i_eff_d2s,idrug,:]
                    popt_occ = popts_occ[i_eff_d2l,i_eff_d2s,idrug,:]
                    d2_slope_act[i_eff_d2l,i_eff_d2s,idr,id] = slope_log(da_eta[id],log_delta,popt_act,type_sigm ='full')
                    d2_slope_occ[i_eff_d2l,i_eff_d2s,idr,id] = slope_log(da_eta[id],log_delta,popt_occ,type_sigm ='full')
                else:
                    d2_slope_act[i_eff_d2l,i_eff_d2s,idr,id]  = slope_log(10**da_eta[id],1/3,ec50_d2,type_sigm ='simple')
                    d2_slope_occ[i_eff_d2l,i_eff_d2s,idr,id]  = slope_log(10**da_eta[id],1/3,ec50_d2,type_sigm ='simple')
                d1_slope[i_eff_d2l,i_eff_d2s,idr,id] = slope_log(10**da_eta[id],3,ec50_d1,type_sigm ='simple')
            eta_act[i_eff_d2l,i_eff_d2s,idr,:] = np.divide(d1_slope[i_eff_d2l,i_eff_d2s,idr,:],(d1_slope[i_eff_d2l,i_eff_d2s,idr,:]+d2_slope_act[i_eff_d2l,i_eff_d2s,idr,:]))
            eta_occ[i_eff_d2l,i_eff_d2s,idr,:] = np.divide(d1_slope[i_eff_d2l,i_eff_d2s,idr,:],(d1_slope[i_eff_d2l,i_eff_d2s,idr,:]+d2_slope_occ[i_eff_d2l,i_eff_d2s,idr,:]))

        drug_concp = np.concatenate([np.asarray([0]),drug_concv])

### Figure 7 & Extended Data Figure 6-8

Plots of asymmetric scaling factor $\tau$ given by the receptor sensitivities


In [ ]:
# Plot asymmetric scaling factors per efficiency:
__,axo = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))
__,axa = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))

for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        for idrug in np.arange(ndrug+1):
            axo[i_eff_d2l,i_eff_d2s].plot(da_eta,eta_occ[i_eff_d2l,i_eff_d2s,idrug,:],color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axa[i_eff_d2l,i_eff_d2s].plot(da_eta,eta_act[i_eff_d2l,i_eff_d2s,idrug,:],color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axo[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
            axa[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
_=[plot_config(ia,'log[DA] nM',r'$\tau_{Occ}$',9,False) for ia in axo.flatten()]
_=[plot_config(ia,'log[DA] nM',r'$\tau_{Act}$',9,False) for ia in axa.flatten()]        


### Figure 7 & Extended Data Figure 6-8

Plots of the equivalent to the  relative reversal learning parameter ( 2 $\tau$ -1) reported by Cools et al, 2009


In [ ]:
# Plot 2tau-1 per efficiency
__,axo = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))
__,axa = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))

for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        for idrug in np.arange(ndrug+1):
            axo[i_eff_d2l,i_eff_d2s].plot(da_eta,2*eta_occ[i_eff_d2l,i_eff_d2s,idrug,:]-1,color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axa[i_eff_d2l,i_eff_d2s].plot(da_eta,2*eta_act[i_eff_d2l,i_eff_d2s,idrug,:]-1,color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axo[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
            axa[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
_=[plot_config(ia,'log[DA] nM',r'2$\tau_{Occ}$-1',9,False) for ia in axo.flatten()]
_=[plot_config(ia,'log[DA] nM',r'2$\tau_{Act}$-1',9,False) for ia in axa.flatten()]        


### Figure 7 & Extended Data Figure 6-8

Plots of the change in the relative reversal learning parameter ( 2 $\tau$ -1) reported by Cools et al, 2009

In [ ]:
# Plot change in asymmetric scaling factors per efficiency:
__,axo = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))
__,axa = plt.subplots(neffs,neffs,figsize=(neffs*4,neffs*4))

for i_eff_d2l in range(neffs):
    for i_eff_d2s in range(neffs):
        for idrug in np.arange(ndrug+1):
            diff_o = (2*eta_occ[i_eff_d2l,i_eff_d2s,idrug,:]-1)-(2*eta_occ[i_eff_d2l,i_eff_d2s,0,:]-1)
            diff_a = (2*eta_act[i_eff_d2l,i_eff_d2s,idrug,:]-1)-(2*eta_act[i_eff_d2l,i_eff_d2s,0,:]-1)

            axo[i_eff_d2l,i_eff_d2s].plot(da_eta,diff_o,color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axa[i_eff_d2l,i_eff_d2s].plot(da_eta,diff_a,color=cm.viridis(idrug/ndrug),
                label=str(np.ceil(drug_concp[idrug]*100)/100)+ ' nM')
            axo[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
            axa[i_eff_d2l,i_eff_d2s].set_title(r'$Eff_{D2l}$='+str(efficacies_p[i_eff_d2l])+r' $Eff_{D2s}$='+str(efficacies_p[i_eff_d2s]),y=.95)
_=[plot_config(ia,'log[DA] nM',r'$\Delta$(2$\tau_{Occ}$)-1',9,False) for ia in axo.flatten()]
_=[plot_config(ia,'log[DA] nM',r'$\Delta$(2$\tau_{Act}$)-1',9,False) for ia in axa.flatten()]  

